# SonarVision - Noise Filtering

Side-scan sonar images carry **speckle**, salt-and-pepper and Gaussian noise. This notebook builds a reusable `preprocess_for_model` function that cleans an image so it is **model-ready**:

1. **Median filter** - removes speckle / salt-and-pepper noise (sonar-typical)
2. **Bilateral filter** - edge-preserving smoothing (keeps target edges sharp)
3. **Light CLAHE** - gentle local contrast (clip 1.2, 16x16 tiles) so flat areas are not over-amplified

> Note: a full-range global stretch is *off by default*. These (synthetic) images are already fairly clean, so aggressive stretching only re-amplifies faint seabed texture into visible grain. Use `stretch=True` only for genuinely dark/noisy imagery.

The dataset archive is loaded **entirely into memory** with `io.BytesIO` / `zipfile`:

- **Local run** (Windows / this 3.10 venv): the zip is read from disk into memory.
- **Google Colab** (no local copy): the same zip is downloaded from Hugging Face with `requests` -> `zipfile.ZipFile(io.BytesIO(response.content))`.

Every filtered image is written to `noise_filtered_training/{train_filtered,val_filtered,test_filtered}/` (“images only”) - ready to zip and upload to Colab for model training. The same `preprocess_for_model` function should be reused at **prediction time**: noise-filter first, then pass to the model.

## 1. Imports

In [ ]:
import io
import zipfile
import requests

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

## 2. The noise-filtering functions

`filter_noise(image, ...)` - pure filtering (Median -> Bilateral -> light CLAHE)

`preprocess_for_model(image, ...)` - the **prediction-time** entry point: loads, filters, resizes to `(target_h, target_w)` and returns a clean uint8 3-channel (BGR) image ready for the model. Pass `as_float=True` to get a `float32` array in `[0, 1]`.

In [ ]:
def filter_noise(
    image,
    median_ksize=5,          # odd int, kernel for median blur (speckle removal)
    bilateral_d=9,           # bilateral filter diameter
    sigma_color=50,          # gray-level sigma (color similarity)
    sigma_space=50,          # spatial sigma
    enable_clahe=True,       # light local contrast enhancement
    clahe_clip=1.2,          # lower clip = gentler contrast (noise-safe)
    clahe_tile=(16, 16),     # larger tiles = broader context, less texture boosting
    stretch=False,           # optional global stretch to [0,255] (off by default)
    stretch_percentile=1.0,  # pixels clipped at each end when stretch=True
    to_float=False,          # True -> return float32 in [0,1]
):
    """Clean a side-scan sonar image: Median -> Bilateral -> light CLAHE.

    Accepts an ndarray (BGR or grayscale) and returns a uint8 BGR 3-channel
    image with the same spatial shape, or float32 [0,1] if to_float=True.
    """
    if isinstance(image, (str, Path)):
        image = cv2.imread(str(image), cv2.IMREAD_UNCHANGED)
        if image is None:
            raise ValueError(f"Could not read image: {image}")

    # Normalize to a single BGR 3-channel array
    if image.ndim == 2:
        bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    elif image.ndim == 3 and image.shape[2] == 4:
        bgr = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    else:
        bgr = np.ascontiguousarray(image)

    # Sonar is effectively grayscale: denoise on a single channel (faster + stable)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # 1) Median blur - kills speckle / salt-and-pepper
    denoised = cv2.medianBlur(gray, median_ksize)

    # 2) Bilateral filter - smooths remaining Gaussian noise, preserves edges
    denoised = cv2.bilateralFilter(
        denoised, bilateral_d, sigma_color, sigma_space
    )

    # 3) Light CLAHE - gentle local contrast (default on, low clip / big tiles)
    if enable_clahe:
        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_tile)
        denoised = clahe.apply(denoised)

    # 4) Optional global stretch to [0,255] - off by default to keep range natural
    if stretch:
        lo = np.percentile(denoised, stretch_percentile)
        hi = np.percentile(denoised, 100 - stretch_percentile)
        denoised = np.clip(
            (denoised.astype(np.float32) - lo) * (255.0 / max(hi - lo, 1)),
            0, 255,
        ).astype(np.uint8)

    out = cv2.merge([denoised] * 3)  # back to BGR 3-channel (model input)
    if to_float:
        return out.astype(np.float32) / 255.0
    return out


def preprocess_for_model(
    image,
    target_w=1024,
    target_h=1024,
    as_float=False,
    **filter_kwargs,
):
    """Prediction-time preprocessing: noise-filter, then resize to model input size.

    Returns a clean uint8 BGR 3-channel image of shape (target_h, target_w, 3),
    or a float32 [0,1] array when as_float=True.
    """
    cleaned = filter_noise(image, to_float=False, **filter_kwargs)
    cleaned = cv2.resize(
        cleaned, (target_w, target_h), interpolation=cv2.INTER_AREA
    )
    if as_float:
        return cleaned.astype(np.float32) / 255.0
    return cleaned

## 3. Configuration

Edit the paths below to match your environment:

- **Local run** (Windows / this notebook): keep `LOCAL_ZIP` and `OUT_DIR`.
- **Google Colab**: the dataset lives outside the repo, so `get_archive()` will **download it from Hugging Face** into memory automatically. Set `OUT_DIR` to something like `/content/noise_filtered_training` (then zip it and save to Drive).

In [ ]:
# ==== Config: edit paths here per environment ====
# Local run (Windows / your venv):
LOCAL_ZIP = Path(r"D:\1. Project Program\1.SIH\dataseta\dataset_final.zip")
OUT_DIR   = Path(r"D:\1. Project Program\1.SIH\dataseta\noise_filtered_training")

# Colab / no local copy - archive is downloaded into memory from Hugging Face:
HF_URL = "https://huggingface.co/datasets/lalitchandra00/sonar_dataset/resolve/main/dataset_final.zip"

# zip inner folder -> output folder (images only)
SUBSETS = {
    "train": "train_filtered",
    "val":   "val_filtered",
    "test":  "test_filtered",
}

# Model input size (resize target)
TARGET_W, TARGET_H = 1024, 1024

## 4. Load the dataset archive (local zip, or Hugging Face into memory)

In [ ]:
def get_archive(local_zip: Path, hf_url: str):
    """Open the dataset zip from memory: local copy when present, otherwise
    download from Hugging Face (same io.BytesIO / zipfile pattern)."""
    if local_zip.exists():
        print(f"Using local zip: {local_zip} ({local_zip.stat().st_size / 1e9:.2f} GB)")
        with open(local_zip, "rb") as fh:
            blob = fh.read()
        archive = zipfile.ZipFile(io.BytesIO(blob))
    else:
        print(f"Local zip not found -> downloading from Hugging Face: {hf_url}")
        response = requests.get(hf_url)  # ~6 GB into memory (Colab OK)
        response.raise_for_status()
        print(f"Downloaded {len(response.content) / 1e9:.2f} GB into memory")
        archive = zipfile.ZipFile(io.BytesIO(response.content))

    print(f"Archive opened. {len(archive.namelist())} entries.")
    return archive


archive = get_archive(LOCAL_ZIP, HF_URL)

## 5. Batch-filter images from the archive

Each subset is filtered with `preprocess_for_model` (denoise + resize to 1024x1024) and written as **images only** into `noise_filtered_training/<subset>_filtered/`. Decoding is done straight from the in-memory zip via `cv2.imdecode`, so no intermediate files are created.

In [ ]:
def filter_archive_to(archive, subset, out_dir, overwrite=False):
    """Filter every image of <subset> from the zip into out_dir (images only)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = f"dataset_final/{subset}/images/"
    members = sorted(
        n for n in archive.namelist()
        if n.startswith(prefix) and n.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    processed = skipped = failed = 0

    for name in tqdm(members, desc=subset):
        out_p = out_dir / Path(name).name
        if out_p.exists() and not overwrite:
            skipped += 1
            continue

        img = cv2.imdecode(
            np.frombuffer(archive.read(name), dtype=np.uint8), cv2.IMREAD_COLOR
        )
        if img is None:
            failed += 1
            continue

        cleaned = preprocess_for_model(img, target_w=TARGET_W, target_h=TARGET_H)
        if not cv2.imwrite(str(out_p), cleaned):
            failed += 1
            continue
        processed += 1

    print(
        f"{subset:6s} -> {out_dir}  | processed={processed} "
        f"skipped={skipped} failed={failed}"
    )
    return processed, skipped, failed


for subset, out_name in SUBSETS.items():
    filter_archive_to(archive, subset, OUT_DIR / out_name)

## 6. Verify output

In [ ]:
print("Noise-filtered output:" + "-" * 43)
grand_total = 0
for out in sorted(OUT_DIR.iterdir()):
    if not out.is_dir():
        continue
    files = [f for f in out.iterdir() if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    grand_total += len(files)
    print(f"{out.name:18s} {len(files):6d} files  {size_mb:8.1f} MB")
print("-" * 43)
print(f"TOTAL: {grand_total} filtered images")

## 7. Before / after (decoded straight from the archive)

In [ ]:
sample = next(
    n for n in archive.namelist()
    if n.startswith("dataset_final/train/images/") and n.lower().endswith(".jpg")
)
orig = cv2.imdecode(np.frombuffer(archive.read(sample), dtype=np.uint8), cv2.IMREAD_COLOR)
cleaned = preprocess_for_model(orig, target_w=TARGET_W, target_h=TARGET_H)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, title, im in zip(axes, ["Original (from zip)", "Noise-filtered (model-ready)"],
                         [orig, cleaned]):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
print("Sample       :", Path(sample).name)
print("Input shape  :", orig.shape, orig.dtype)
print("Output shape :", cleaned.shape, cleaned.dtype)
print("Output range : [%d, %d]" % (cleaned.min(), cleaned.max()))
plt.tight_layout()
plt.show()

## 8. (Optional) Zip the filtered dataset for easy upload to Colab / Drive

Run this only when you need it - it packs `noise_filtered_training/` into a single `noise_filtered_training.zip` so you can drag one file into Google Colab (or Google Drive) instead of thousands of images.

In [ ]:
ZIP_OUT = OUT_DIR.with_suffix(".zip")
print(f"Zipping {OUT_DIR} -> {ZIP_OUT} ...")
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_STORED) as zf:
    for img_p in tqdm(sorted(OUT_DIR.rglob("*.jpg")), desc="zip"):
        zf.write(img_p, img_p.relative_to(OUT_DIR.parent))
print(f"Done: {ZIP_OUT.stat().st_size / 1e9:.2f} GB")